# Setup

In [15]:
%load_ext jupyter_ai_magics

import os
import sys

# ----------------------------------------------------
# 1. Google API Key Configuration (Securely load from environment variables)
# ----------------------------------------------------
# Check if the 'GOOGLE_API_KEY' environment variable is set.
# If not set, issue a warning or terminate the process.
google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key is None:
    print("Error: Environment variable 'GOOGLE_API_KEY' is not set.", file=sys.stderr)
    print("Please set your API key as an environment variable before running this code.", file=sys.stderr)
    # If the API key is strictly required, you can stop execution here:
    # sys.exit(1)
else:
    # Setting it in os.environ allows Google API client libraries to access it.
    # While redundant if already in the environment, explicit assignment ensures availability.
    os.environ["GOOGLE_API_KEY"] = google_api_key
    print("✅ Google API key successfully loaded from environment variables.")


# ----------------------------------------------------
# 2. Loading Jupyter AI Magics
# ----------------------------------------------------
# '%load_ext' is a magic command that only works in Jupyter Notebook or IPython environments.
# To prevent 'NameError' when running as a standard Python script, 
# we wrap it in a try-except block for better robustness.
try:
    # Use the IPython global function get_ipython() to execute the magic command.
    # (This will raise a NameError if not in a Jupyter/IPython environment)
    get_ipython().run_line_magic('load_ext', 'jupyter_ai_magics')
    print("✅ Jupyter AI Magics successfully loaded.")
except NameError:
    print("⚠️ Warning: Not in an IPython/Jupyter environment. Skipping '%load_ext jupyter_ai_magics'.", file=sys.stderr)
except Exception as e:
    print(f"❌ An unexpected error occurred while loading Jupyter AI Magics: {e}", file=sys.stderr)

# Once AI Magics is loaded, you can use it within Jupyter cells as usual.
# Example:
# %%ai chatgpt "Hello, world!"

The jupyter_ai_magics extension is already loaded. To reload it, use:
  %reload_ext jupyter_ai_magics
✅ Google API key successfully loaded from environment variables.
The jupyter_ai_magics extension is already loaded. To reload it, use:
  %reload_ext jupyter_ai_magics
✅ Jupyter AI Magics successfully loaded.


In [ ]:
import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)
import numpy as np
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import HTML, display

InteractiveShell.ast_node_interactivity = "all"
import glob
# import warnings
# warnings.filterwarnings("ignore")
import hashlib
import os
import re
import time
from datetime import datetime

import xlwings as xw

# Import

In [4]:
# Define the folder path
folder_path = os.path.expanduser("~/Downloads_win")

# Get all files that start with 'export' and end with '.xlsx'
export_files = [
    f for f in os.listdir(folder_path) if f.startswith("export") and f.endswith(".xlsx")
]

# Check if any matching files exist
if not export_files:
    raise FileNotFoundError("No 'export*.xlsx' files found in the specified folder.")

# Find the latest file based on modification time
latest_file = max(
    export_files, key=lambda f: os.path.getmtime(os.path.join(folder_path, f))
)

# Build the full path to the latest file
latest_file_path = os.path.join(folder_path, latest_file)

# Read the Excel file using pandas
df = pd.read_excel(latest_file_path, engine="openpyxl")

print(f"Loaded file: {latest_file_path}")
df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: '/home/mrsmmori/Downloads_win'

# Cleaning

In [ ]:
def normalize_aum(value):
    if pd.isna(value):
        return pd.Series([None, None])

    match = re.match(r"([A-Z]+)\s([\d\.]+)([MBk])", str(value).strip())
    if not match:
        return pd.Series([None, None])

    currency, amount, unit = match.groups()
    amount = float(amount)

    # 単位をミリオンに変換
    if unit == "B":
        amount *= 1000
    elif unit == "M":
        amount *= 1
    elif unit == "k":
        amount /= 1000  # 例えば 874.96k → 0.87496

    return pd.Series([currency, round(amount, 2)])


df[["AUM_Currency", "AUM_normalized"]] = df["AUM"].apply(normalize_aum)

print(df[["AUM", "AUM_Currency", "AUM_normalized"]].head())

           AUM AUM_Currency  AUM_normalized
0  USD 183.80M          USD          183.80
1   USD 57.42M          USD           57.42
2    USD 2.42B          USD         2420.00
3  USD 840.00k          USD            0.84
4    USD 1.05M          USD            1.05


In [ ]:
df["Risk Rating"] = df["Risk Rating"].replace("-", np.nan)
df["Risk Rating"] = pd.to_numeric(df["Risk Rating"], errors="coerce")
df["Dividend Yield (%)"] = pd.to_numeric(df["Dividend Yield (%)"], errors="coerce")
df["Annual Expense Ratio"] = df["Annual Expense Ratio"].replace("-", np.nan)
df["Annual Expense Ratio"] = pd.to_numeric(df["Annual Expense Ratio"], errors="coerce")
df["Dividend Yield (%)"] = df["Dividend Yield (%)"].round(2)
df["Risk Rating"] = df["Risk Rating"].round(2)
df["AUM_normalized"] = df["AUM_normalized"].round(0).astype("Int64")

# 20250902

In [ ]:
df_usd = df[df["Currency"] == "USD"]
(
    df_usd.sort_values(by="AUM_normalized", ascending=False)
    .head(100)
    .sort_values(by="Dividend Yield (%)", ascending=False)
    .head(5)
)

,Product Name,Exchanges,Stock Code,Type,Currency,Risk Rating,Asset Class,Specialist Sector,Geographic Sector,AUM,Annual Expense Ratio,Dividend Yield (%),Dividend Frequency,Underlying Index,Min. RSP Amt,SRS RSP Enabled,3 YR Sharpe Ratio_x000d_\n,AUM_Currency,AUM_normalized
4166,JPMorgan Nasdaq Equity Premium Income ETF,NASDAQ,JEPQ,ETFs,USD,8.0,Multi-Asset,General,US,USD 29.75B,0.35,11.22,Monthly,S&P 500 TR USD,SGD 50.00,-,-,USD,29750
4152,JPMorgan Equity Premium Income ETF,NYSE,JEPI,ETFs,USD,8.0,Equity,General,US,USD 41.37B,0.35,8.40,Monthly,S&P 500 TR USD,SGD 50.00,-,-,USD,41370
8212,iShares Broad USD High Yield Corporate Bond ETF,BATS,USHY,ETFs,USD,4.0,Fixed Income,High Yield,US,USD 25.29B,0.08,6.72,Monthly,ICE BofA US Broad Market TR USD,-,-,-,USD,25290
4179,Janus Henderson AAA CLO ETF,NYSE,JAAA,ETFs,USD,2.0,Fixed Income,Securitised Credit,Global,USD 24.85B,0.20,5.84,Monthly,Bloomberg US Agg Bond TR USD,-,-,-,USD,24850
4177,JPMorgan Ultra-Short Income ETF,NYSE,JPST,ETFs,USD,2.0,Fixed Income,Short Duration,Global,USD 33.08B,0.18,4.77,Monthly,Bloomberg US Agg Bond TR USD,-,-,-,USD,33080
